Notebook per la creazione e l'analisi del dataset scaricato da PDB: PL- v.2020R1

1. estrazione informazioni da INDEX_general_PL.2020R1.ist
2. pulizia csv 
3. aggiunta Smile dall' sdf

4. visualizzazione




In [ ]:
import os
import re #regex library
import csv
import pandas as pd
import numpy as np

#graphein 
import networkx as nx
from functools import partial

import py3Dmol
from graphein.protein.config import ProteinGraphConfig
from graphein.protein.graphs import construct_graph
from graphein.protein.visualisation import plotly_protein_structure_graph
from graphein.protein.edges.distance import add_distance_threshold

#biopython
from rdkit import Chem

#ignore warinings
import warnings
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings("ignore")

#dataframe info
datapath = "data\\PDB"

## **Analisi e Scraping Dataset**


### **Estrazione informazioni INDEX**

In [39]:
# read and extract info from INDEX_general_PL
filename = "INDEX_general_PL.2020R1.lst"
filepath = os.path.join(datapath, filename)

if not os.path.exists(filepath):
    raise FileNotFoundError(f"{filename} file not found path[{filepath}]")

#read file
with open(filepath, "r", encoding="utf-8") as f:
    lines = [
        ln.strip()
        for ln in f
        if ln.strip() and not ln.lstrip().startswith("#")
    ]
    
#regex [PDB_code, resolution, release_year, binding_data[type,compare,unit], reference, ligand_name, comments]
regex= re.compile(r"(\w+)\s+"+                              #pdb code
                  r"([\d.]+|NMR)\s+"+                       #resolution
                  r"(\d{4})\s+"+                            #release year
                  r"(\w+)([=<~]{0,2})([\d\.]+)(\w+)\s+"     #binding data
                  r"//\s[\w\.]+\s+"                         #file path reference (ignored)
                  r"\(([^)]+?)\)"                           #ligand name
                  r"\s*(.*)"                                #comments
                  )

parsed_index = []
skipped = 0
for ln in lines:
    match = regex.match(ln)
    if match:
        (pdb, res, year, btype, comp, val, unit, ligand, comment) = match.groups()
        
        parsed_index.append({
            "PDB_code": pdb,
            "resolution": res,
            "release_year": year,
            "binding_type": btype,
            "compare": comp,
            "value": val,
            "unit": unit,
            "ligand_name": ligand,
            "comments": comment.strip()
        })
    else:
        skipped += 1
        #print("SKIPPED:", ln)
print(f"Parsed: {len(parsed_index)} | Skipped: {skipped}")

#salvo csv
csv_filename = "parsed_index.csv"
csv_filepath = os.path.join(datapath, csv_filename)

with open(csv_filepath, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=parsed_index[0].keys())
    writer.writeheader()
    writer.writerows(parsed_index)
       


Parsed: 19037 | Skipped: 0


### **Pulizia dataframe**

In [40]:
#dataframe info
csv_filename = "parsed_index.csv"
csv_filepath = os.path.join(datapath, csv_filename)

df = pd.read_csv(csv_filepath)
initial_entries = len(df)
#df.info()
#df.describe()

#CLEANING OF DATA
#remove NMR entries
df = df[df["resolution"] != "NMR"]
#remove <, >, ~ from compare and value
df = df[df["compare"] == "="]
#convert resolution to float
df["resolution"] = df["resolution"].astype(np.float32)
#convert release_year to int
df["release_year"] = df["release_year"].astype(int)
#convert value to float
df["value"] = df["value"].astype(np.float32)

#remove entries with resolution > 2.5 Å
df = df[df["resolution"] <= 2.5]

#remove comments with "no ligand" or "no binding" ... (forse meglio togliere tutte le righe con commenti)
invalid_comments=["incomplete ligand",
                  "covalent complex",
                  "covalent bond",
                  "isomer",
                  "Different protein in assay",
                  "Different ligand in assay",
                  "Nonstandard assay",
                  ]
df = df[~df["comments"].str.contains("|".join(invalid_comments), case=False, na=False)]
remaining_comments = df["comments"].unique()
#print(remaining_comments)

#Associate correct decade to release_year
def change_year(year):
    if year <= 2000:
        return "1981-2000"
    elif year <= 2010:
        return "2001-2010"
    else:
        return "2011-2019"
df['release_year'] = df['release_year'].apply(change_year)


#ANALYSIS OF BINDING DATA
binding_counts = df["binding_type"].value_counts()
binding_percentages = df["binding_type"].value_counts(normalize=True) * 100
for btype, count in binding_counts.items():
    percentage = binding_percentages[btype]
    print(f"{btype}\t: {count} entries ({percentage:.2f}%)")
    
#dizionario per la conversione in Molarità (M)
unit_multiplier = {
    'mM': 1e-3,
    'uM': 1e-6,
    'nM': 1e-9,
    'pM': 1e-12,
    'fM': 1e-15
}

#conversione e aggiunta colonna molarità [value_M]
df["value_M"] = df["value"] * df["unit"].map(unit_multiplier).fillna(1) 

# uniformare le 3 misure calcolando il logaritmo negativo (pAffinity)
# questo trasforma Kd, Ki e IC50 in una singola scala pKd / pKi / pIC50
df=df[df["value_M"] > 0]  #rimuovo valori non validi (zero o negativi)
df['pAffinity'] = -np.log10(df['value_M']).astype(np.float32).round(4)

# OPXIONALE: IC50 è meno affidabile come misura di affinità possiamo scartarlo e mantenere solo Kd e Ki (35% abbiamo molti meno dati)
#df = df[df['binding_type'].isin(['Kd', 'Ki'])]

#print(df.head(10))

#risultati cleaning:
print("\nData Cleaning Results:")
entries_removed_percentage = (initial_entries - len(df)) / initial_entries * 100
print(f"Entries before cleaning: {initial_entries}")
print(f"Entries after cleaning: {len(df)}")
print(f"Removed: {initial_entries - len(df)} ({entries_removed_percentage:.2f}%)")


IC50	: 4931 entries (36.62%)
Kd	: 4853 entries (36.04%)
Ki	: 3682 entries (27.34%)

Data Cleaning Results:
Entries before cleaning: 19037
Entries after cleaning: 13466
Removed: 5571 (29.26%)


### **Aggiunta SMILE**

In [41]:
#add SMILE column to dataframe
df['SMILE'] = None

def get_clean_smiles(smile_path):
    #load molecola da file SDF
    suppl = Chem.SDMolSupplier(smile_path, removeHs=True, sanitize=True)
    
    if len(suppl) == 0 or suppl[0] is None:
        print(f"Error loading molecule from {smile_path}")
        return None
        
    mol = suppl[0] #prendo la prima molecola (dovrebbe essere l'unica)
    try:
        # corregge aromaticità, valenze e tipi di legame
        Chem.SanitizeMol(mol)
        
        #SMILES Canonico e Isomerico (mantiene la chiralità @ o @@)
        smile = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
        return smile
    except:
        return None

print("\nExtracting SMILES from SDF files...")
for row in df.itertuples():
    sdf_filename = f"{row.PDB_code}_ligand.sdf"
    smile_path = os.path.join(datapath, "P-L",row.release_year, row.PDB_code , sdf_filename)
    
    if not os.path.exists(smile_path):
        print(f"Impossible to find SDF file for [{row.PDB_code}] -> Path does not exist: {smile_path}")
        
        continue
    
    df.at[row.Index, 'SMILE'] = get_clean_smiles(smile_path)
    
print("removing entries with invalid SMILES...")
df = df[df['SMILE'].notna()]

print("\nSMILES extraction completed. Sample data:")   
df.head()


Extracting SMILES from SDF files...
Error loading molecule from data\PDB\P-L\1981-2000\1lvk\1lvk_ligand.sdf
Error loading molecule from data\PDB\P-L\2011-2019\3vjs\3vjs_ligand.sdf
Error loading molecule from data\PDB\P-L\2011-2019\3vjt\3vjt_ligand.sdf
removing entries with invalid SMILES...

SMILES extraction completed. Sample data:


,PDB_code,resolution,release_year,binding_type,compare,value,unit,ligand_name,comments,value_M,pAffinity,SMILE
0,2tpi,2.1,1981-2000,Kd,=,49.000,uM,2-mer,NaN,4.900000e-05,4.3098,CC[C@H](C)[C@H]([NH3+])C(=O)N[C@H](C(=O)[O-])C...
2,4tln,2.3,1981-2000,Ki,=,190.000,uM,LNO,NaN,1.900000e-04,3.7212,CC(C)C[C@H]([NH3+])C(=O)NO
5,1rnt,1.9,1981-2000,Kd,=,6.500,uM,2GP,NaN,6.500000e-06,5.1871,Nc1[nH]c(=O)c2ncn([C@@H]3O[C@H](CO)[C@@H](O)[C...
7,4ts1,2.5,1981-2000,Kd,=,11.600,uM,TYR,NaN,1.160000e-05,4.9355,[NH3+][C@@H](Cc1ccc(O)cc1)C(=O)[O-]
8,4tmn,1.7,1981-2000,Ki,=,0.068,nM,0PK,NaN,6.800000e-11,10.1675,CC(C)C[C@H](N[P@](=O)([O-])[C@H](Cc1ccccc1)NC(...


In [42]:
#SALVATAGGIO CSV

#Drop useles colums and save cleaned dataset
cleaned_csv_filename = "PL_cleaned_index.csv"
cleaned_csv_filepath = os.path.join(datapath, cleaned_csv_filename)
df.drop(columns=["binding_type","compare", "value","unit", "value_M","ligand_name", "comments"], inplace=True)
df.to_csv(cleaned_csv_filepath, index=False)

## **Data Retrival**

In [ ]:
#dataframe info
csv_filename = "PL_cleaned_index.csv"
csv_filepath = os.path.join(datapath, csv_filename)
df = pd.read_csv(csv_filepath)

# PDB ID dal dataframe locale
pdb_id = df['PDB_code'].iloc[12]
print(f"Proteina selezionata: {pdb_id}")

#visualizzazione della proteina con py3Dmol
def visualizza_proteina(pdb_id):
    # Crea il visualizzatore
    view = py3Dmol.view(query=f'pdb:{pdb_id}')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    view.zoomTo()
    return view.show()
visualizza_proteina(pdb_id)

#creazione e visualizzazione del grafo
def create_protein_graph(pdb_code: str) -> nx.Graph:
    edge_funcs = [partial(add_distance_threshold, long_interaction_threshold=0, threshold=8.0)]
    config = ProteinGraphConfig(edge_construction_functions=edge_funcs)
    print(f"Costruzione grafo 3D per {pdb_code} in corso...")
    return construct_graph(config=config, pdb_code=pdb_code)

def visualize_protein_3D(g: nx.Graph):
    fig = plotly_protein_structure_graph(
        g,
        colour_edges_by="kind",
        colour_nodes_by="seq_position",
        node_size_multiplier=1
    )
    fig.show()


G_protein = create_protein_graph(pdb_id)
print(f"\nNodi (Amminoacidi): {G_protein.number_of_nodes()}")
print(f"Archi (Interazioni): {G_protein.number_of_edges()}")
visualize_protein_3D(G_protein)
    

Proteina selezionata: 4er1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Costruzione grafo 3D per 4er1 in corso...



Nodi (Amminoacidi): 330
Archi (Interazioni): 2036
